# 메인 LangGraph — 질문 하나 → Tool 1개 → 실DB → QueryOutput (11/05 B07·B08, B09)

`src/rag/school/main_graph.py`의 `run_main(query, deps, registry)`를 **실제 공유 Neo4j**로 실행한다.

- 노드: normalize_input → classify_scope → classify_intent → select_tool → execute_tool → merge_and_validate → evaluate_evidence → compose_answer | partial_or_insufficient.
- 오늘 registry에는 **실제로 연결된 `action_search`만** 있다. disease/legal/crossdoc을 고르면 호출하지 않고(0회) 연결 오류로 남긴다. 다른 Tool로 대신 답하지 않는다.
- 추가 Tool 재진입·총 3회 한도는 11/06 작업이다(오늘은 1회 종료).
- 가짜 Tool로 하는 분기 검사는 `tests/rag/school/test_school_main_graph.py`. 여기는 실DB 경로와 실패 분기 증빙이다.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

from index.school.dictionaries import load_dictionaries
from rag.school.graph import default_deps

deps = default_deps()            # driver·database·embedder. driver는 이 노트북이 소유하고 마지막 셀에서 닫는다.
deps["books"] = load_dictionaries()
DRIVER, DATABASE = deps["driver"], deps["database"]
ALLOWED = ["moe_school_manual_v3", "gyeonggi_school_guide_2025", "kdca_measles_2026",
           "kdca_diagnosis_reporting_2026", "moleg_infectious_disease_act_21521"]
print("Neo4j 접속 확인(접속 정보는 출력하지 않음) / embedder:", type(deps["embedder"]).__name__)

Neo4j 접속 확인(접속 정보는 출력하지 않음) / embedder: OpenAIEmbeddings


## 메인 1-Tool 실행 (M3 게이트)

In [2]:
from rag.school.contract_v2_local import validate_query_output
from rag.school.main_graph import run_main
from rag.school.validation_local import verify_against_db

query = {"question": "학교에서 유증상 학생이 발견되면 담임교사와 보건교사는 각각 무엇을 확인하나요?",
         "region": "GYEONGGI", "allowed_document_ids": ALLOWED, "request_id": "demo-20261105-01"}
out = run_main(query, deps)
validate_query_output(out, query)
report = verify_against_db(out, DRIVER, DATABASE)
print("response_type:", out["response_type"], "/ status:", out["status"], "/ tools_used:", out["tools_used"],
      "/ search_rounds:", out["search_rounds"])
print("Finding", len(out["findings"]), "/ Citation", len(out["citations"]), "/ action_ids", len(out["action_ids"]))
print("최종 출력 DB 대조:", "통과" if not report["bad_citations"] and not report["bad_findings"] else report)
print("최종 출력 키:", sorted(out))        # graph_paths·domain 없음
print()
print(out["answer"])

response_type: DOMAIN / status: ANSWERED / tools_used: ['action_search'] / search_rounds: 1
Finding 12 / Citation 11 / action_ids 8
최종 출력 DB 대조: 통과
최종 출력 키: ['action_ids', 'answer', 'citations', 'findings', 'response_type', 'search_rounds', 'status', 'tools_used', 'warnings']

원문에서 확인한 내용입니다.
- [교육부 매뉴얼 PDF 54쪽 · 원문 배치: 대응 제1단계] 원문: “ 감염병 (의심)환자가 교내에서 혼자 이동하는 것은 원칙적으로 금지  이동 시에는 담임(담당) 교사가 동행하며 2m 정도의 거리 유지  담당 교사는 담임교사 권장 (만약 임신부 등 감염병 고위험군인 경우 다른 교사로 대체)” [1]
- [교육부 매뉴얼 PDF 197쪽] 원문: “7) 교사 및 학부모의 역할 ∙ 정기적인 예방접종 권장 ∙ 보건(담당)교사 및 담임교사는 학교 내 (의심)환자 발생에 따라 교육(지원)청에 보고 및 조치함” [2]
- [교육부 매뉴얼 PDF 57쪽 · 원문 배치: 대응 제1단계] 원문: “ 보호자에게 의료기관 진료 여부를 확인하고 미진료 시 진료를 받도록 하며, 이때 출결 관련 필요 서류 및 학교 홈페이지의 관련 서식 이용 방법 안내  감염병 증상으로 결석한 사실을 인지한 즉시 보건(담당)교사에게 연락” [3]
- [교육부 매뉴얼 PDF 207쪽] 원문: “∙ 보건(담당)교사 및 담임교사는 학교 내 (의심)환자 발생에 따라 보고 및 조치함 ∙ 가정과 학교에서 노출가능성이 있는 사람들에게 증상 감시를 권고함 ∙ 성홍열 의심증상자는 즉시 의료기관으로 진료를 권고함” [4]
- [경기도 가이드북 PDF 166쪽 · 대응 제1단계] 원문이 제시한 담당 예시: 담임교사 — 감염병 유증상자를 발견한다. [5]
- [교육부 매뉴얼 PDF 36쪽, 모

### 노드 실행 기록 (request_id 상관관계, 실제 호출 1회)

In [3]:
steps = [t for t in deps["main_trace"] if t["step"] != "tool_results"]
display(pd.DataFrame(steps))
tool_results = deps["main_trace"][-1]["results"]
print("ToolResult 수:", len(tool_results), "/ GraphPath는 ToolResult에만:", len(tool_results[0]["graph_paths"]))

,step,request_id,region,allowed,response_type,confidence,intent,secondary,tool,registered,status,findings,citations,graph_paths,seconds,error,unresolved
0,normalize_input,demo-20261105-01,GYEONGGI,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,classify_scope,NaN,NaN,NaN,DOMAIN,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,classify_intent,NaN,NaN,NaN,NaN,NaN,ACTION,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,select_tool,NaN,NaN,NaN,NaN,NaN,NaN,NaN,action_search,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,execute_tool,demo-20261105-01,NaN,NaN,NaN,NaN,NaN,NaN,action_search,NaN,OK,12.0,11.0,8.0,3.46,NaN,NaN
5,merge_and_validate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0,11.0,NaN,NaN,NaN,0.0
6,evaluate_evidence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ANSWERED,NaN,NaN,NaN,NaN,NaN,0.0


ToolResult 수: 1 / GraphPath는 ToolResult에만: 8


## 분기·실패 검사 (T05·T10·T11, 미등록 Tool)

In [4]:
def show(q, registry=None, **extra):
    o = run_main({"question": q, "region": "GYEONGGI", "allowed_document_ids": ALLOWED, **extra}, deps, registry)
    tool = next((t.get("tool") for t in deps["main_trace"] if t["step"] == "select_tool"), None)
    return {"question": q, "response_type": o["response_type"], "selected": tool, "status": o["status"],
            "tools_used": o["tools_used"], "rounds": o["search_rounds"], "findings": len(o["findings"]),
            "note": (o["warnings"] or [""])[0][:80]}

cases = [show("안녕하세요, 이 서비스로 무엇을 할 수 있나요?"),
         show("오늘 점심 메뉴 추천해 줘."),
         show("우리 반 학생이 열이 나고 몸에 발진이 있어요. 성홍열인지 판단해 주세요."),
         show("홍역의 잠복기는 며칠인가요?"),
         show("성홍열도 학교장이 보건소에 신고해야 하나요?"),
         show("보호자에게 연락해 진료를 요청하는 일을 교육부 매뉴얼과 경기도 가이드북은 각각 어떻게 설명하나요?"),
         show("학년부장은 능동감시에서 무엇을 하나요?")]
display(pd.DataFrame(cases))
assert all(c["rounds"] == 0 for c in cases[:6])          # 일반·범위 밖·미등록 Tool은 호출 0회
assert cases[-1]["rounds"] == 1 and cases[-1]["tools_used"] == ["action_search"]   # T05 1회 종료

,question,response_type,selected,status,tools_used,rounds,findings,note
0,"안녕하세요, 이 서비스로 무엇을 할 수 있나요?",GENERAL,NaN,ANSWERED,[],0,0,
1,오늘 점심 메뉴 추천해 줘.,OUT_OF_SCOPE,NaN,INSUFFICIENT_EVIDENCE,[],0,0,학교 감염병 대응 자료와 관련된 질문으로 보이지 않는다.
2,우리 반 학생이 열이 나고 몸에 발진이 있어요. 성홍열인지 판단해 주세요.,OUT_OF_SCOPE,NaN,INSUFFICIENT_EVIDENCE,[],0,0,진단·질병 판정은 서비스 범위 밖이다. 증상으로 질병을 판정하지 않으며 의료기관 진료가 필요하다.
3,홍역의 잠복기는 며칠인가요?,DOMAIN,disease_search,ERROR,[],0,0,"선택한 Tool 'disease_search'이 메인 그래프에 연결되지 않아 호출하지 않았다(연결 오류, 호출 0회). 다른 Tool로 대신 답"
4,성홍열도 학교장이 보건소에 신고해야 하나요?,DOMAIN,legal_search,ERROR,[],0,0,"선택한 Tool 'legal_search'이 메인 그래프에 연결되지 않아 호출하지 않았다(연결 오류, 호출 0회). 다른 Tool로 대신 답하지"
5,보호자에게 연락해 진료를 요청하는 일을 교육부 매뉴얼과 경기도 가이드북은 각각 어떻게 설명하나요?,DOMAIN,crossdoc_graph_search,ERROR,[],0,0,"선택한 Tool 'crossdoc_graph_search'이 메인 그래프에 연결되지 않아 호출하지 않았다(연결 오류, 호출 0회). 다른 Too"
6,학년부장은 능동감시에서 무엇을 하나요?,DOMAIN,action_search,ANSWERED,[action_search],1,12,원문 인용 Finding은 검색된 원문을 그대로 옮긴 것이다. 해석·적용 판단은 포함하지 않는다.


In [5]:
# T11: Tool 실행 장애는 호출 1회로 기록되고 ERROR. 근거 없음(NO_EVIDENCE)과 구분된다.
def broken(tool_input):
    raise ConnectionError("Neo4j 연결 끊김(시뮬레이션)")

failed = show("담임교사는 무엇을 하나요?", {"action_search": broken})
no_docs = show("담임교사는 무엇을 하나요?") | {}
empty = run_main({"question": "담임교사는 무엇을 하나요?", "region": "GYEONGGI", "allowed_document_ids": []}, deps)
display(pd.DataFrame([failed, {"question": "allowed=[]", "response_type": empty["response_type"],
                               "status": empty["status"], "tools_used": empty["tools_used"],
                               "rounds": empty["search_rounds"], "findings": len(empty["findings"]),
                               "note": empty["warnings"][0][:80]}]))
assert failed["status"] == "ERROR" and failed["rounds"] == 1
assert empty["status"] == "INSUFFICIENT_EVIDENCE" and empty["search_rounds"] == 1

,question,response_type,selected,status,tools_used,rounds,findings,note
0,담임교사는 무엇을 하나요?,DOMAIN,action_search,ERROR,[action_search],1,0,Tool 실행 실패: ConnectionError: Neo4j 연결 끊김(시뮬레이션)
1,allowed=[],DOMAIN,NaN,INSUFFICIENT_EVIDENCE,[action_search],1,0,허용 문서 목록이 비어 있어 검색하지 않았다.


## 질문셋 v1.1 40문항 — 라우팅·상태·호출 기록

`notebooks/school/평가결과/golden_eval_main.py`로 실행한 결과를 읽는다. 미연결 Tool로 라우팅된 문항은 상태를 채점하지 않는다(호출 0회 ERROR가 정상).

In [6]:
rows = pd.DataFrame(json.loads((ROOT / "data/interim/school/eval_main_final.json").read_text(encoding="utf-8")))
display(rows[["id", "split", "want_route", "got_route", "route_ok", "want_status", "got_status", "status_ok",
              "rounds", "hit", "total", "n_find"]])
graded = rows[rows["status_ok"].notna()]
print("라우팅 일치:", int(rows["route_ok"].sum()), "/", len(rows),
      "| 연결된 경로 status 일치:", int(graded["status_ok"].astype(bool).sum()), "/", len(graded),
      "| 최대 호출 수:", int(rows["rounds"].max()))

,id,split,want_route,got_route,route_ok,want_status,got_status,status_ok,rounds,hit,total,n_find
0,S-Q01,dev,action_search,action_search,True,ANSWERED,ANSWERED,True,1,3,5,12
1,S-Q02,dev,action_search,action_search,True,ANSWERED,ANSWERED,True,1,1,4,12
2,S-Q03,eval,legal_search,legal_search,True,PARTIAL,ERROR,None,0,0,2,0
3,S-Q04,dev,crossdoc_graph_search,crossdoc_graph_search,True,ANSWERED,ERROR,None,0,0,3,0
4,S-Q05,dev,action_search,action_search,True,ANSWERED,ANSWERED,True,1,2,2,12
5,S-Q06,eval,action_search,action_search,True,ANSWERED,ANSWERED,True,1,0,1,8
6,S-Q07,dev,action_search,action_search,True,ANSWERED,ANSWERED,True,1,2,2,12
7,S-Q08,dev,action_search,action_search,True,ANSWERED,ANSWERED,True,1,2,2,12
8,S-Q09,eval,action_search,action_search,True,ANSWERED,ANSWERED,True,1,1,1,12
9,S-Q10,dev,action_search,action_search,True,ANSWERED,ANSWERED,True,1,0,3,12


라우팅 일치: 40 / 40 | 연결된 경로 status 일치: 34 / 35 | 최대 호출 수: 1


In [ ]:
# 11/05 검색 적합성 개선을 기존 커널에도 반영한다.
import importlib
import rag.school.action_tool as action_tool_module
import rag.school.main_graph as main_graph_module

importlib.reload(action_tool_module)
importlib.reload(main_graph_module)
run_main = main_graph_module.run_main
deps["trace"] = []

QUESTION = "유행성각결막염 걸렸을 때도 전파차단 위한 휴업 해야하나요?"
REGION = "GYEONGGI"      # 또는 "NATIONAL"

out = run_main({"question": QUESTION, "region": REGION, "allowed_document_ids": ALLOWED}, deps)

print("status:", out["status"], "/ tools:", out["tools_used"], "/ rounds:", out["search_rounds"])
print("질병·단계 힌트:", deps["trace"][-1]["plan"]["diseases"], deps["trace"][-1]["plan"]["stages"])
print("Finding / Citation:", len(out["findings"]), len(out["citations"]))
print()
print(out["answer"])
print("\n--- 근거 ---")
DOC_NAMES = {"moe_school_manual_v3": "교육부 매뉴얼", "gyeonggi_school_guide_2025": "경기도 가이드북"}
for n, c in enumerate(out["citations"], 1):
    printed = f"(인쇄 {c['printed_page']}쪽)" if c["printed_page"] else ""
    print(f"[{n}] {DOC_NAMES.get(c['document_id'], c['document_id'])} PDF {c['pdf_page']}쪽 {printed}")
    print("    " + c["excerpt"].replace("<br>", " / "))


status: ANSWERED / tools: ['action_search'] / rounds: 1
질병·단계 힌트: ['EKC'] None
Finding / Citation: 2 2

원문에서 확인한 내용입니다.
- [교육부 매뉴얼 PDF 104쪽] 원문: “ 전파 차단을 위한 휴업 인플루엔자, 수두, 유행성이하선염, 수족구병, 유행성각결막염 등의 학생 빈발 감염병에 대해서는 전파차단을 위한 휴업은 권장하지 않음(단, 방역당국의 역학조사 결과 휴업을 권고하는 경우에는 실시)” [1]
- [경기도 가이드북 PDF 236쪽] 원문: “- 인플루엔자, 수두, 유행성이하선염, 수족구병, 유행성각결막염 등의 학생 빈발 감염병에 대해서는 전파 차단을 위한 휴업은 권장하지 않음(단, 방역당국의 역학조사 결과 휴업을 권고하는 경우에는 실시) - 신종감염병과 같이 치명률이 높은 감염병으로 방역당국의 권고가 있는 경우에는 학교 내에 단 한 명의 환자가” [2]
참고: 원문 인용 Finding은 검색된 원문을 그대로 옮긴 것이다. 해석·적용 판단은 포함하지 않는다.

--- 근거 ---
[1] 교육부 매뉴얼 PDF 104쪽 (인쇄 90쪽)
     전파 차단을 위한 휴업
인플루엔자, 수두, 유행성이하선염, 수족구병, 유행성각결막염 등의 학생 빈발 감염병에 대해서는 전파차단을 위한
휴업은 권장하지 않음(단, 방역당국의 역학조사 결과 휴업을 권고하는 경우에는 실시)
[2] 경기도 가이드북 PDF 236쪽 (인쇄 226쪽)
    - 인플루엔자, 수두, 유행성이하선염, 수족구병, 유행성각결막염 등의 학생 빈발 감염병에 대해서는 전파 차단을
위한 휴업은 권장하지 않음(단, 방역당국의 역학조사 결과 휴업을 권고하는 경우에는 실시)
- 신종감염병과 같이 치명률이 높은 감염병으로 방역당국의 권고가 있는 경우에는 학교 내에 단 한 명의 환자가


In [7]:
DRIVER.close()